In [1]:
!pip install streamlit pyngrok sentence-transformers pyspark==3.5.1 numpy pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 66.8 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [30]:
%%writefile app.py
import streamlit as st
import numpy as np
import pandas as pd
import math
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.pipeline import PipelineModel

BASE = "/content/drive/MyDrive/ProjectBigData/05 artifacts"

PIPELINE_PATH = f"{BASE}/hospital_price_pipeline"
CLEAN_PATH    = f"{BASE}/hospital_prices_clean"
ALIGNED_PATH  = f"{BASE}/hospital_prices_aligned"
EMB_PATH      = f"{BASE}/desc_emb.npy"

# ==========================================================
# Streamlit Title
# ==========================================================
st.title("Hospital Procedure Price Predictor")
st.write("Enter a procedure description to estimate discounted cash prices across hospitals.")

# ==========================================================
# Load Artifacts
# ==========================================================
@st.cache_resource
def load_spark():
    return SparkSession.builder.appName("PricePredict").getOrCreate()

spark = load_spark()

@st.cache_resource
def load_pipeline():
    return PipelineModel.load(PIPELINE_PATH)

@st.cache_resource
def load_desc_emb():
    return np.load(EMB_PATH)

@st.cache_resource
def load_embed_model():
    return SentenceTransformer("all-MiniLM-L6-v2")

@st.cache_resource
def load_df_clean():
    return spark.read.parquet(CLEAN_PATH)

@st.cache_resource
def load_df_aligned():
    return spark.read.parquet(ALIGNED_PATH)

desc_emb = load_desc_emb()
embed_model = load_embed_model()
df_clean = load_df_clean()
df_aligned = load_df_aligned()
pipeline_model = load_pipeline()

# Anatomy mapping
ANATOMY = {
    "knee": ["knee", "patellar", "patella", "lower extremity", "leg"],
    "hip": ["hip", "pelvis"],
    "spine": ["spine", "lumbar", "thoracic", "cervical"],
    "shoulder": ["shoulder", "rotator cuff"],
    "elbow": ["elbow"],
    "wrist": ["wrist"],
    "hand": ["hand"],
    "ankle": ["ankle"],
    "foot": ["foot"],
    "brain": ["brain", "head", "cranial"],
    "abdomen": ["abdomen", "abdominal"],
    "heart": ["heart", "cardiac"],
    "lung": ["lung", "chest", "thorax"]
}

def detect_anatomy(text):
    text = text.lower()
    for region, terms in ANATOMY.items():
        if any(t in text for t in terms):
            return region
    return None

def same_anatomy(desc, region):
    if region is None:
        return False
    desc = desc.lower()
    for t in ANATOMY[region]:
        if t in desc:
            return True
    return False

# ==========================================================
# Prediction Function (same logic as v4)
# ==========================================================
def predict_price_v4(user_text, top_k=15):

    user_emb = embed_model.encode([user_text])
    scores = cosine_similarity(user_emb, desc_emb)[0]
    top_idx = scores.argsort()[::-1][:top_k]

    candidates = []
    for idx in top_idx:
        row = df_aligned.filter(F.col("seq_id") == int(idx)).first()
        if row:
            candidates.append({
                "seq_id": int(idx),
                "description": row.description,
                "similarity": float(scores[idx]),
                "code_2": row.code_2,
                "code_2_type": row.code_2_type,
                "code_1": row.code_1,
                "code_1_type": row.code_1_type,
                "code_3": row.code_3,
                "setting": row.setting
            })

    # Imaging-aware filter
    imaging_terms = ["mri", "ct", "scan", "ultrasound", "x-ray", "xray"]
    if any(t in user_text.lower() for t in imaging_terms):
        img_only = [c for c in candidates if str(c["code_2"]).startswith("7")]
        if img_only:
            candidates = img_only

    # Anatomy filtering
    anat = detect_anatomy(user_text)
    if anat:
        anat_only = [c for c in candidates if same_anatomy(c["description"], anat)]
        if anat_only:
            candidates = anat_only

    # Outpatient first, else fallback
    outpatient = [c for c in candidates if c["setting"] and c["setting"].lower() == "outpatient"]
    selected = outpatient[0] if outpatient else candidates[0]

    # Fetch hospital rows for selected CPT + setting
    hospitals = (
        df_clean.filter(F.col("code_2") == selected["code_2"])
                .filter(F.col("setting") == selected["setting"])
                .select("hospital_name", "gross_price")
                .collect()
    )

    rows = []
    for h in hospitals:
        gp = float(h.gross_price)
        rows.append({
            "description": selected["description"],
            "hospital_name": h.hospital_name,
            "code_2": selected["code_2"],
            "code_2_type": selected["code_2_type"],
            "code_1": selected["code_1"],
            "code_1_type": selected["code_1_type"],
            "code_3": selected["code_3"],
            "setting": selected["setting"],
            "gross_price": gp,
            "log_gross": math.log(gp)
        })

    spark_df = spark.createDataFrame(rows)
    pred_df = pipeline_model.transform(spark_df)
    pred_df = pred_df.withColumn("pred_cash", F.exp(F.col("prediction")))

    # Convert to pandas
    dfp = (
        pred_df
        .select("hospital_name", "pred_cash")
        .orderBy("hospital_name")
        .toPandas()
    )

    # Convert hospital names to Title Case
    dfp["hospital_name"] = dfp["hospital_name"].str.title()

    # Rename columns
    dfp.rename(columns={
        "hospital_name": "Hospital",
        "pred_cash": "Predicted Price ($)"
    }, inplace=True)

    # Round to 2 decimals + format with $
    dfp["Predicted Price ($)"] = dfp["Predicted Price ($)"].round(2)
    dfp["Predicted Price ($)"] = dfp["Predicted Price ($)"].apply(lambda x: f"${x:,.2f}")

    return dfp, selected, candidates[:5]


# ==========================================================
# Streamlit UI
# ==========================================================
user_input = st.text_input("Enter a medical procedure description:")

if user_input:
    st.subheader("Predicted Discounted Cash Prices")
    result, selected, matches = predict_price_v4(user_input)

    # st.write(f"**Matched Procedure:** {selected['description']}")
    # st.write(f"**CPT:** {selected['code_2']}")
    # st.write(f"**Setting:** {selected['setting']}")
    # st.write(f"**Similarity:** {selected['similarity']:.3f}")

    st.table(result)

    # st.subheader("Top Matching Procedures")
    # st.json(matches)


Overwriting app.py


In [6]:
from pyngrok import ngrok

In [ ]:
from google.colab import userdata

ngrok_auth = userdata.get('ngrok_auth_token')

In [7]:

ngrok.set_auth_token("ngrok_auth")


**Execute below to view app**

In [31]:
!pkill streamlit
!streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false &>/dev/null &


In [33]:
# !pkill ngrok


In [34]:
from pyngrok import ngrok
public_url = ngrok.connect(8501)
public_url


<NgrokTunnel: "https://homological-gasmetophytic-jeni.ngrok-free.dev" -> "http://localhost:8501">